# 01 Load Excel Files

* Author: Jeremiah Hansen
* Last Updated: 2/27/2026

This notebook will load data into the `LOCATION` and `ORDER_DETAIL` tables from Excel files.

This currently does not use Snowpark File Access as it doesn't yet work in Notebooks. So for now we copy the file locally first.

In [ ]:
# Import python packages
import sys
import logging

# Set up the logger
logger_name = 'demo_logger'
logger = logging.getLogger(logger_name)
logger.setLevel(logging.INFO)

# Set default values for debugging
notebook_name = '01_load_excel_files.ipynb'
database_name = 'DEMO_DB'
schema_name = 'DEV_SCHEMA'
role_name = 'DEMO_ROLE'

# Override values with passed notebook arguments
if sys.argv[0].endswith('.ipynb'):
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--database-name', type=str)
    parser.add_argument('--schema-name', type=str)
    parser.add_argument('--role-name', type=str)
    args, args_unknown = parser.parse_known_args()

    notebook_name = parser.prog  # same as argv[0]
    database_name = args.database_name or database_name
    schema_name = args.schema_name or schema_name
    role_name = args.role_name or role_name

# Get a Snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Set the default database and schema for the following cells
session.use_schema(f"{database_name}.{schema_name}")

# Set the role
# Needed when running EXECUTE NOTEBOOK PROJECT directly (since it ignores session context and uses the user's default role)
session.use_role(role_name)

# Get details about the current state
current_state_df = session.sql(f"""
        SELECT OBJECT_CONSTRUCT(
            'current_user', CURRENT_USER(),
            'current_role', CURRENT_ROLE(),
            'current_secondary_roles', PARSE_JSON(CURRENT_SECONDARY_ROLES()),
            'current_database', CURRENT_DATABASE(),
            'current_schema', CURRENT_SCHEMA()
        )::STRING AS session_context;
    """).collect()

logger.info(f"Begin executing notebook {notebook_name}", extra = {'logger_name': logger_name})
logger.info(f"Using parameters database: {database_name}, schema: {schema_name}, role: {role_name}", extra = {'logger_name': logger_name})
logger.info(f"Using session context {current_state_df[0]['SESSION_CONTEXT']}", extra = {'logger_name': logger_name})

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
print('Excel parsing libraries loaded (zipfile + xml)')

In [ ]:
%%sql -r dataframe_1
-- Temporary solution to load in the metadata, this should be replaced with a directy query to a directory table (or a metadata table)
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/order_detail.xlsx' AS STAGE_FILE_PATH, 'order_detail' AS WORKSHEET_NAME, 'ORDER_DETAIL' AS TARGET_TABLE
UNION
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/location.xlsx', 'location', 'LOCATION';

## Create a function to load Excel worksheet to table

Create a reusable function to load an Excel worksheet to a table in Snowflake.

Note: Until we can use scoped URLs in Notebooks, via the `BUILD_SCOPED_FILE_URL()` function, we need to temporarily copy the file to a temp stage and then process from there.

In [ ]:
from snowflake.snowpark.files import SnowflakeFile
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
import pandas as pd

session.sql("CREATE TEMP STAGE IF NOT EXISTS temp_excel_stage").collect()

def _parse_xlsx(file_bytes, worksheet_name):
    """Parse an xlsx file using zipfile and xml (no openpyxl needed)"""
    NS = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
    
    with zipfile.ZipFile(BytesIO(file_bytes)) as zf:
        # Read shared strings
        shared_strings = []
        if 'xl/sharedStrings.xml' in zf.namelist():
            ss_tree = ET.parse(zf.open('xl/sharedStrings.xml'))
            for si in ss_tree.iter(f'{NS}si'):
                text_parts = []
                for t in si.iter(f'{NS}t'):
                    if t.text:
                        text_parts.append(t.text)
                shared_strings.append(''.join(text_parts))
        
        # Find the target worksheet
        wb_tree = ET.parse(zf.open('xl/workbook.xml'))
        sheet_names = []
        for sheet in wb_tree.iter(f'{NS}sheet'):
            sheet_names.append(sheet.get('name'))
        
        if worksheet_name not in sheet_names:
            raise ValueError(f"Worksheet '{worksheet_name}' not found. Available: {sheet_names}")
        
        sheet_idx = sheet_names.index(worksheet_name) + 1
        sheet_path = f'xl/worksheets/sheet{sheet_idx}.xml'
        
        # Also check rels for actual sheet path
        if sheet_path not in zf.namelist():
            for name in zf.namelist():
                if name.startswith('xl/worksheets/') and name.endswith('.xml'):
                    sheet_path = name
                    break
        
        sheet_tree = ET.parse(zf.open(sheet_path))
        
        # Parse rows
        rows = []
        for row in sheet_tree.iter(f'{NS}row'):
            row_data = []
            for cell in row.iter(f'{NS}c'):
                cell_type = cell.get('t')
                value_elem = cell.find(f'{NS}v')
                if value_elem is not None and value_elem.text is not None:
                    if cell_type == 's':
                        row_data.append(shared_strings[int(value_elem.text)])
                    elif cell_type == 'b':
                        row_data.append(bool(int(value_elem.text)))
                    else:
                        try:
                            val = float(value_elem.text)
                            row_data.append(int(val) if val == int(val) else val)
                        except ValueError:
                            row_data.append(value_elem.text)
                else:
                    inline_str = cell.find(f'{NS}is')
                    if inline_str is not None:
                        t_elem = inline_str.find(f'{NS}t')
                        row_data.append(t_elem.text if t_elem is not None else None)
                    else:
                        row_data.append(None)
            rows.append(row_data)
    
    if not rows:
        return pd.DataFrame()
    
    columns = rows[0]
    data = rows[1:]
    # Pad rows to match column count
    max_cols = len(columns)
    data = [r + [None] * (max_cols - len(r)) for r in data]
    return pd.DataFrame(data, columns=columns)

def load_excel_worksheet_to_table(session, stage_path, worksheet_name, target_table):
    """Load an Excel worksheet from a stage path into a Snowflake table"""
    
    filename = stage_path.split('/')[-1]
    
    session.sql(f"""
        COPY FILES INTO @temp_excel_stage
        FROM {stage_path}
    """).collect()
    
    with SnowflakeFile.open(f'@temp_excel_stage/{filename}', 'rb') as f:
        file_bytes = f.read()
    
    df = _parse_xlsx(file_bytes, worksheet_name)
    
    df = df.astype(str).replace('None', None)
    
    snowpark_df = session.create_dataframe(df)
    snowpark_df.write.mode("overwrite").save_as_table(target_table)
    
    logger.info(f"Loaded {len(df)} rows from '{worksheet_name}' to {target_table}", extra = {'logger_name': logger_name})

## Process all Excel worksheets

Loop through each Excel worksheet to process and call our `load_excel_worksheet_to_table_local()` function.

In [ ]:
# Process each file from the sql_get_spreadsheets cell above
files_to_load = dataframe_1
for index, excel_file in files_to_load.iterrows():
    print(f"Processing Excel file {excel_file['STAGE_FILE_PATH']}")
    load_excel_worksheet_to_table(session, excel_file['STAGE_FILE_PATH'], excel_file['WORKSHEET_NAME'], excel_file['TARGET_TABLE'])

logger.info(f"Finish executing notebook {notebook_name}", extra = {'logger_name': logger_name})

### Debugging

In [ ]:
%%sql -r dataframe_2
DESCRIBE TABLE LOCATION;
SELECT * FROM LOCATION;
SHOW TABLES;